# Análise de Consumo de Energia - Notebook
Este notebook lê um arquivo CSV do S3 e realiza diversas manipulações com `pandas` conforme os requisitos.

In [2]:
pip install boto3

   ---------------------------------------- 0.0/13.8 MB ? eta -:--:--
   ------ --------------------------------- 2.4/13.8 MB 12.2 MB/s eta 0:00:01
   ------------- -------------------------- 4.7/13.8 MB 11.9 MB/s eta 0:00:01
   -------------------- ------------------- 7.1/13.8 MB 11.8 MB/s eta 0:00:01
   ---------------------------- ----------- 9.7/13.8 MB 11.8 MB/s eta 0:00:01
   ---------------------------------- ----- 12.1/13.8 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------- 13.8/13.8 MB 11.4 MB/s eta 0:00:00
  Attempting uninstall: botocore
    Found existing installation: botocore 1.34.69
    Uninstalling botocore-1.34.69:
      Successfully uninstalled botocore-1.34.69
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.12.3 requires botocore<1.34.70,>=1.34.41, but you have botocore 1.39.4 which is incompatible.


In [3]:
import boto3
import pandas as pd
from io import StringIO

## Leitura do arquivo CSV do S3

In [4]:
session = boto3.Session(profile_name='teste')
s3 = session.client('s3')

bucket_name = 'isabellabucket123'
object_key = 'basedados.csv'

csv_obj = s3.get_object(Bucket=bucket_name, Key=object_key)
body = csv_obj['Body'].read().decode('utf-8')
df = pd.read_csv(StringIO(body), sep=';')

df.head()

,Cidade,Campus,Ano,Mês,Quantidade em kWh
0,Ituiutaba,Pontal,2023,Janeiro,52150
1,Ituiutaba,Pontal,2023,Fevereiro,44800
2,Ituiutaba,Pontal,2023,Março,62300
3,Ituiutaba,Pontal,2023,Abril,56000
4,Ituiutaba,Pontal,2023,Maio,54250


## Conversão de mês por extenso para número

In [5]:
meses_map = {
    'janeiro': 1, 'fevereiro': 2, 'março': 3, 'abril': 4,
    'maio': 5, 'junho': 6, 'julho': 7, 'agosto': 8,
    'setembro': 9, 'outubro': 10, 'novembro': 11, 'dezembro': 12
}
df['Mês_Num'] = df['Mês'].str.lower().map(meses_map)

## Criação da coluna de data

In [ ]:
df['Data'] = pd.to_datetime(df['Ano'].astype(str) + '-' + df['Mês_Num'].astype(str) + '-01')

## Filtro: São Paulo e 2023

In [6]:
df_filtrado = df[(df['Cidade'] == 'São Paulo') & (df['Ano'] == 2023)]
df_filtrado

,Cidade,Campus,Ano,Mês,Quantidade em kWh,Mês_Num


## Agregação: Consumo total por campus

In [7]:
agregado = df.groupby('Campus')['Quantidade em kWh'].sum().reset_index()
agregado

,Campus,Quantidade em kWh
0,Araras,497560
1,Educaço Física,521430
2,Fazendas,650489
3,Glória,393050
4,Pontal,681800
5,Santa Mônica,4754400
6,Umuarama,3286200


## Condicional: consumo alto (> 10000 kWh)

In [8]:
df['Consumo Alto'] = df['Quantidade em kWh'].apply(lambda x: 'Sim' if x > 10000 else 'Não')

## Conversão de kWh para MWh

In [9]:
df['Quantidade em MWh'] = df['Quantidade em kWh'] / 1000

## Nome do campus em maiúsculas

In [10]:
df['Campus_MAIUSCULO'] = df['Campus'].str.upper()
df.head()

,Cidade,Campus,Ano,Mês,Quantidade em kWh,Mês_Num,Consumo Alto,Quantidade em MWh,Campus_MAIUSCULO
0,Ituiutaba,Pontal,2023,Janeiro,52150,1,Sim,52.15,PONTAL
1,Ituiutaba,Pontal,2023,Fevereiro,44800,2,Sim,44.80,PONTAL
2,Ituiutaba,Pontal,2023,Março,62300,3,Sim,62.30,PONTAL
3,Ituiutaba,Pontal,2023,Abril,56000,4,Sim,56.00,PONTAL
4,Ituiutaba,Pontal,2023,Maio,54250,5,Sim,54.25,PONTAL
